a) None을 반환한다
re.match는 문자열의 시작부터 패턴이 일치하는지를 검사합니다. d는 숫자를 +는 패턴의 반복을 의미하는데 text는 한글로 시작하므로 None을 반환합니다


b)2026-05-06
re.search는 문자열 전체에서 패턴과 일치하는 부분을 찾습니다. d는 숫자 {4}와 {2}는 정확히 네자리, 두자리가 온다는 것을 의미하므로 text에서 2026,05,06을 찾습니다. .group()을 이용해 이를 문자열로 반환합니다.

c) ['2026-05-06',2026-05-18] 
re.findall은 re.search와 다르게 문자열 전체에 있는 모든 패턴을 리스트로 반환합니다.

d)[('2026', '05', '06'), ('2026', '05', '18')]
괄호를 넣으면 캡쳐그룹을 지정한다는 의미입니다. 캡쳐그룹 두개 이상이 패턴에 있을 경우 패턴만을 뽑아서 튜플로 저장합니다. 

e) ['2026-05-06', '2026-05-18']
괄호에 ?:이 있을 경우 비 캡쳐그룹이 되어서 c와 같은 결과를 반환합니다.

추가질문:
캡쳐그룹을 사용하면 문자열 전체가 아닌 그룹에 해당하는 부분만 튜플로 반환합니다. 따라서 d는 괄호 없이 연, 월, 일이 쪼개진 튜플을 반환하지만 비캡쳐그룹인 c와 e는 온전한 날짜 문자열 리스트를 반환합니다.

a)[T]!
re.sub은 패턴을 [T]로 대체합니다. <.+>는 <부터 시작해서 아무 문자가 한번 이상 나오고 가장 마지막에 >로 끝나는 부분을 패턴으로 한다는 의미입니다. 따라서 <b>안녕...</i>가 [T] 가 되고 남은 !가 뒤에 붙습니다

b)[T]안녕[T] [T]세상[T]!
?가 앞에 붙으면 가장 먼저 >로 끝나는 부분까지 매칭하는 게으른 매칭입니다. 때문에 태그가 있던 자리가 [T]로 바뀝니다.

c) [T]안녕[T] [T]세상[T]!
<[^>]+>는 < 로 시작해서 >가 아닌 문자가 한번 이상 나온 후 >로 끝나는 부분까지를 패턴으로 인식한다. 때문에 b와 같은 결과를 반환합니다

d)수강생<30>명, 조교<3>명
패턴으로 숫자 덩어리를 캡쳐그룹으로 하였습니다. 치환문자열 r"<\1>"은 앞에서 찾은 숫자덩어리를 \1에 넣어서 30을 찾았을때는 <30>, 3을 찾았을때는 3으로 바꾸어 반환합니다.

e)수강생<\x01>명, 조교<\x01>명
원시문자열이 아닌 일반문자열 안에서 \1은 아스키코드 1번문자 \x01로 해석됩니다. 때문에 정규표현식 치환이 제대로 이루어지지 않습니다.

추가질문:
1: 탐욕적 매칭은 가장 긴 문자열까지 매칭하는 반면 게으른 매칭은 가장 짧은 문자열을 탐색합니다.

2: d는 원시문자열을 사용하여 \1이 올바르게 정규표현식에 전달되지만 e는 일반문자열을 사용하여 정규표현식에 전달되기전에 파이썬에 의해 8진수 이스케이프 문자로 해석됩니다.

In [ ]:
#사용한 llm 대화 내역
#https://claude.ai/share/86283968-b13a-4ec3-a6df-28024947e280

import re

def clean_post(post: str) -> str:
    # 1. URL 제거 (http:// 또는 https://로 시작하여 공백 전까지) → 공백 한 칸으로 치환
    post = re.sub(r'https?://\S+', ' ', post)
    
    # 2. HTML 태그(<...>) 제거 → 빈 문자열로
    post = re.sub(r'<[^>]+>', '', post)
    
    # 3. 이메일 → [이메일], 전화번호 → [전화] 마스킹
    post = re.sub(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', '[이메일]', post)
    post = re.sub(r'\d{2,4}-\d{3,4}-\d{4}', '[전화]', post)
    
    # 4. 멘션(@\w+)과 해시태그(#\w+) → 공백 한 칸으로 치환
    post = re.sub(r'@\w+', ' ', post)
    post = re.sub(r'#\w+', ' ', post)
    
    # 5. 한글 자음/모음(U+3131–U+3163) 1개 이상 연속 → 빈 문자열로 제거
    post = re.sub(r'[\u3131-\u3163]+', '', post)
    
    # 6. 공백 문자(\s+) → 공백 한 칸으로 정리, 앞뒤 strip
    post = re.sub(r'\s+', ' ', post)
    post = post.strip()
    
    return post


def extract_hashtags(post: str) -> list[str]:
    # # 뒤에 한글, 영문 대소문자, 숫자의 연속체만 해시태그로 인정
    # 반환 시 # 제외
    return re.findall(r'#([A-Za-z0-9가-힣]+)', post)

def analyze_posts(posts: list[str]) -> dict:
    posts_n = len(posts)
    
    total_length = 0
    hashtag_counter = {}
    masked_count = 0
    
    for post in posts:
        # masked_count: re.subn의 치환 횟수 활용
        post_step, n1 = re.subn(r'https?://\S+', ' ', post)
        post_step, n2 = re.subn(r'<[^>]+>', '', post_step)
        post_step, n3 = re.subn(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', '[이메일]', post_step)
        post_step, n4 = re.subn(r'\d{2,4}-\d{3,4}-\d{4}', '[전화]', post_step)
        post_step, n5 = re.subn(r'@\w+', ' ', post_step)
        post_step, n6 = re.subn(r'#\w+', ' ', post_step)
        post_step, n7 = re.subn(r'[\u3131-\u3163]+', '', post_step)
        post_step, n8 = re.subn(r'\s+', ' ', post_step)
        cleaned = post_step.strip()
        
        # 이메일 + 전화번호 마스킹 건수만 집계
        masked_count += n3 + n4
        
        # 평균 글자 수 누적
        total_length += len(cleaned)
        
        # 해시태그 빈도 집계 (원본에서 추출)
        for tag in extract_hashtags(post):
            hashtag_counter[tag] = hashtag_counter.get(tag, 0) + 1
    
    avg_length_after_clean = round(total_length / posts_n, 2) if posts_n > 0 else 0.0
    
    # 빈도 내림차순 정렬
    hashtag_counts = dict(sorted(hashtag_counter.items(), key=lambda x: x[1], reverse=True))
    
    return {
        "posts_n": posts_n,
        "avg_length_after_clean": avg_length_after_clean,
        "hashtag_counts": hashtag_counts,
        "masked_count": masked_count,
    }


In [ ]:

posts: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ "
    "자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15강. "    
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
    " 여러 공백과\n\n\n줄바꿈이 많은 텍스트 ",
    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
]

for p in posts:
    c_post = clean_post(p)
    print(c_post)

for p in posts:
    h = extract_hashtags(p)
    print(h)

a = analyze_posts(posts)
print(a)


오늘 수업 진짜 재밌었음!! 감사 자료:
팀플 어디서 모이지 카톡
중요: 다음 시험 범위는 1-15강. 문의는 [이메일] ([전화])로!
여러 공백과 줄바꿈이 많은 텍스트
진짜 좋다
['파이썬']
['DCCP2026', '팀플']
[]
[]
['파이썬', '추천']
{'posts_n': 5, 'avg_length_after_clean': 19.4, 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}, 'masked_count': 2}


오늘 수업 진짜 재밌었음!! 감사 자료:
팀플 어디서 모이지 카톡
중요: 다음 시험 범위는 1-15강. 문의는 [이메일] ([전화])로!
여러 공백과 줄바꿈이 많은 텍스트
진짜 좋다


['파이썬']
['DCCP2026', '팀플']
[]
[]
['파이썬', '추천']


{'posts_n': 5, 'avg_length_after_clean': 19.4, 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}, 'masked_count': 2}

a)설명
url제거의 경우 r'https?://\S+,'를 이용하여 http로 시작하고 s? 즉 s가 있을수도 있고 없을수도 있으며 그 뒤에 ://이 등장하고 \S+는 공백이 아닌 모든 문자들을 의미합니다.
즉 http(s)://로 시작해서 공백 전까지를 ' '로 바꿉니다.

html 태그 제거는 r'<[^>]+>를 이용하여 <로 시작하고 >가 아닌 문자열을 포함해서 >가 처음 등장할 때 까지를 패턴으로 인식하여 ''로 교체합니다.

이메일과 전화번호 마스킹은 [a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,} 즉 알파벳 대소문자와 숫자, 특수문자와 @ 그리고 도메인주소 조합으로 된 패턴을 [이메일]로 바꾸고 숫자 2-4자리 하이픈 3-4자리 하이픈 4자리로 된 패턴을 [전화]로 바꿉니다.


멘션과 해시태그 치환은 알파벳 대소문자와 숫자, 언더바를 의미하는 \w를 이용하여 @과 # 뒤에오는 \w들을 패턴으로 잡아서 ' '로 치환합니다

한글 자모음이 연속으로 나타나는 경우는 한글 자모음의 유니코드인 u3131-u3163까지의 연속된 패턴을 공백으로 치환합니다.

공백문자 정리는 \s를 이용하여 공백, 엔터 탭 등을 패턴으로 잡아 공백 한칸으로 치환합니다. 또한 기본 메서드인 strip을 이용하여 문자열 앞, 뒤의 공백을 정리합니다.


a를 진행할 때 순서를 따라야 하는 이유는 다음과 같습니다.
예를 들어 3번과 4번의 순서가 뒤바뀌면 이메일을 잡기 전에 이메일의 @뒷부분이 태그로 잡혀버립니다. 또한 공백치환을 마지막에 하지 않을 경우 다른 처리를 하고 남은 공백이 최종 출력에 남아있게 되는 문제 역시 생깁니다. 때문에 패턴을 잡을 때 자세한 패턴 부터 잡아야 합니다.



b) 설명
findall과 캡쳐그룹을 이용하여 #뒤에 한글영어숫자의 연속을 캡쳐그룹으로 하여 해당 패턴만을 반환합니다.

c) 설명
게시물의 수는 len함수를 이용하여 얻습니다.

post_step, nx 으로 a)의 과정을 반복함과 동시에 subn을 이용하여 post_step에는 정리된 문자열을, nx에는 각 스텝에 사용된 치환 횟수를 저장합니다.
마스킹 카운트는 n3과 n4에서만 발생하므로 둘을 더한 값이 됩니다.

정제된 게시물의 평균 글자 수는 한 post에 대해 정제한 문자열의 길이를 total_length에 더하여 이를 post의 개수로 나누어 구합니다. 이떄 round함수를 이용하여 둘째자리까지 반올림합니다.

해시태그 카운트는 해시태그 카운터라는 빈 딕셔너리를 만들어서 .get메서드를 이용해 처음 등장한 태그라면 딕셔너리에 추가하고 이미 등장했다면 밸류에 1을 추가하였다. 그뒤 sorted함수를 이용하여 내림차순 정렬을 하였다


